In [3]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)

device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
print("device:", device)

model_name = "distilbert/distilgpt2"
save_path = "./distilgpt2-digital"

max_length = 256
train_limit = 20000
eval_limit = 2000


/home/hyun/miniconda3/envs/al/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda:3


In [4]:
# 위키텍스트 데이터셋 불러오기 (학습/검증/테스트 포함)
# Load the WikiText dataset (already split into train/validation/test)
raw = load_dataset("wikitext", "wikitext-2-raw-v1")

# 어떤 데이터 분할(train/validation/test)이 있는지 확인
# Check which splits are included in the dataset
print("loaded splits:", raw)

loaded splits: DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})


In [5]:
# 토크나이저 준비
# Prepare the tokenizer for the model

# 사전학습된 모델(distilgpt2)과 짝이 맞는 토크나이저를 불러온다
# Load the tokenizer that matches the pretrained model
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=True
)

# GPT 계열 모델은 기본적으로 pad_token이 정의되어 있지 않다
# GPT-style models do not define a padding token by default
if tokenizer.pad_token is None:
    # 배치 학습 시 문장 길이를 맞추기 위해 pad_token이 필요하므로 문장의 끝을 의미하는 eos_token을 pad_token으로 대신 사용한다
    # Use eos_token as pad_token to enable batching with padding
    tokenizer.pad_token = tokenizer.eos_token

# 현재 사용 중인 padding 토큰과 문장 끝 토큰을 출력하여 확인한다
# Print pad_token and eos_token for verification
print(
    "pad_token:", tokenizer.pad_token,
    "| eos_token:", tokenizer.eos_token
)

pad_token: <|endoftext|> | eos_token: <|endoftext|>


In [6]:
# 토크나이징 함수 정의
# Define a tokenization function
def tokenize_fn(examples):
    # truncation=True 옵션과 max_length를 설정해 너무 긴 문장은 잘라준다
    # Use truncation=True and max_length to cut off overly long sentences
    return tokenizer(examples["text"], truncation=True, max_length=max_length)

In [9]:
# train/eval 데이터셋 만들기
# Create train/evaluation datasets

# 모델 파라미터를 업데이트하는 데 사용되는 학습용 데이터셋
# Training dataset used to update model parameters
train_ds = raw["train"].select(
    range(min(len(raw["train"]), train_limit))
).map(
    tokenize_fn,
    batched=True,
    remove_columns=raw["train"].column_names
)

# 평가(검증) 단계에서 성능을 측정하기 위한 데이터셋
# Evaluation dataset used to measure model performance
eval_ds = raw["validation"].select(
    range(min(len(raw["validation"]), eval_limit))
).map(
    tokenize_fn,
    batched=True,
    remove_columns=raw["validation"].column_names
)

# 데이터셋 객체를 PyTorch 텐서 형태로 자동 변환하도록 설정
# Set the dataset format so that outputs are returned as PyTorch tensors
train_ds.set_format("torch")
eval_ds.set_format("torch")

# 실제로 학습과 평가에 사용되는 데이터 개수를 출력
# Print the number of samples used for training and evaluation
print("train size:", len(train_ds), "eval size:", len(eval_ds))

# train 데이터셋의 첫 번째 샘플이 가지고 있는 키들을 출력
# Print the keys (fields) of the first training sample
print("train sample keys:", train_ds[0].keys())

Map: 100%|██████████| 20000/20000 [00:01<00:00, 12606.58 examples/s]

train size: 20000 eval size: 2000
train sample keys: dict_keys(['input_ids', 'attention_mask'])


In [13]:
# 배치를 만들면서 padding과 labels(정답 토큰)을 자동으로 생성
# Automatically create padded batches and generate labels (target tokens)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [16]:
# 모델 로드, 언어모델을 사전학습 가중치로 불러와 서버 GPU로 옮기는 과정
# Load the pretrained language model and move it to the server GPU
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

# 모델이 정상적으로 로드되었는지 확인
# Confirm that the model has been successfully loaded
print("MODEL LOADED")

MODEL LOADED


In [18]:
# training arguments 설정
# Set training arguments
training_args = TrainingArguments(
    # 학습 과정에서 생성되는 결과물을 저장할 디렉토리
    # Directory to save training outputs and logs
    output_dir="./distilgpt2_ft_results",

    # 언제 평가(eval)를 수행할지 결정
    # Define when to run evaluation (e.g., every epoch)
    eval_strategy="epoch",

    # 모델 체크포인트 저장 여부
    # Strategy for saving model checkpoints
    save_strategy="no",

    # 옵티마이저 학습률
    # Learning rate for the optimizer
    learning_rate=5e-5,

    # GPU 하나당 학습 배치 크기
    # Training batch size per GPU
    per_device_train_batch_size=8,

    # 평가 시 사용할 배치 크기
    # Evaluation batch size per GPU
    per_device_eval_batch_size=8,

    # 전체 학습 데이터셋을 몇 번 반복할지
    # Number of training epochs
    # (1 for testing/debugging, 2–3 recommended for proper fine-tuning)
    num_train_epochs=1,

    # 가중치 감쇠를 통한 정규화
    # Weight decay for regularization
    weight_decay=0.01,

    # 학습 로그를 몇 step마다 출력할지
    # Log training metrics every N steps
    logging_steps=50,

    # 로그를 외부 서비스(W&B, TensorBoard 등)로 보낼지 여부
    # Whether to report logs to external services
    report_to="none",

    # fp16(혼합 정밀도) 사용 여부, GPU가 있으면 활성화
    # Enable fp16 (mixed precision) training if a GPU is available
    fp16=torch.cuda.is_available(),
)

In [20]:
# trainer 생성: 학습에 필요한 모든 요소를 하나로 묶어 Trainer 객체를 구성
# Create the Trainer by assembling the model, datasets, and training configurations
trainer = Trainer(
    # 학습 및 평가에 사용할 언어 모델
    # The language model to be trained and evaluated
    model=model,

    # 학습 전략 및 하이퍼파라미터 설정
    # Training strategy and hyperparameters
    args=training_args,

    # 학습에 사용할 데이터셋
    # Dataset used for training
    train_dataset=train_ds,

    # 평가(검증)에 사용할 데이터셋
    # Dataset used for evaluation
    eval_dataset=eval_ds,

    # 토크나이저 (padding, 디코딩 등에 사용)
    # Tokenizer used for padding, decoding, and text processing
    tokenizer=tokenizer,

    # 배치 생성 규칙 (padding 및 labels 자동 생성)
    # Rules for creating batches (automatic padding and label generation)
    data_collator=data_collator,
)

# Trainer가 정상적으로 준비되었는지 확인
# Confirm that the Trainer is ready
print("TRAINER READY")

TRAINER READY


/tmp/ipykernel_852366/4019889095.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [26]:
#fine tunning 실행
trainer.train()
print("FINETUNING DONE")

Epoch,Training Loss,Validation Loss
1,3.572900,3.503544


FINETUNING DONE


In [28]:
# 디지털 모델 저장
# Save the digital model
import os

# save_path는 원하는 디렉토리 이름으로 수정
# save_path can be changed to any desired directory name
save_path = "./distilgpt2-digital"

# 해당 경로에 모델이 이미 존재하는지 확인
# Check whether the model already exists at the given path
if not os.path.exists(save_path):
    print("Saving the digital model...")
    
    # 학습된 모델 가중치와 설정(config) 저장
    # Save the trained model weights and configuration
    model.save_pretrained(save_path)
    
    # 토크나이저 관련 파일 저장 (vocab, merges, special tokens 등)
    # Save tokenizer files (vocab, merges, special tokens, etc.)
    tokenizer.save_pretrained(save_path)
else:
    # 이미 동일한 경로에 모델이 존재할 경우
    # If a model already exists at the specified path
    print("A model already exists at:", save_path)


A model already exists at: ./distilgpt2-digital


In [30]:
# 앞에서 저장해 둔 fine-tuned 디지털 모델을 다시 메모리(GPU/CPU)로 불러와 추가 실험에서 바로 사용할 수 있도록 함
# Load the previously saved fine-tuned digital model into memory (GPU/CPU) for further experiments
print("Loading the saved digital model...")

# 저장된 경로에서 모델 구조(config)와 학습된 가중치를 함께 로드
# Load the model configuration and trained weights from the saved path
base_model = AutoModelForCausalLM.from_pretrained(save_path).to(device)

# 모델이 정상적으로 로드되었는지 확인
# Confirm that the model has been successfully loaded
print("Loaded from:", save_path)

Loading the saved digital model...
Loaded from: ./distilgpt2-digital
